# 05 · Key families: do both sides hash the same key?

Sampling (`sample` = fraction or fixed N) picks rows by the **normalised key**. Production Left and
Right backends compile each family's `left_transform` / `right_transform` into their own SQL dialect.
Distributed Right computes sampled statistics in Spark SQL; local pulls recheck an already-filtered CSV.
A suitable family makes different representations (`007` vs `7`, timestamp vs compact date) hash alike;
a wrong family can select different rows and create false Stage 2 differences.

This notebook now executes the pair's real `sample` plan in both backend queries. Fraction sampling uses
the shared deterministic hash predicate; fixed N is selected once by Left and reused verbatim on Right.

This notebook needs no run. It reads one pair from `CONFIG`, pulls only the key columns of a small
window from both sides, and shows:

1. what each key family does, in Python and in the warehouse SQL, on values you choose;
2. matched / Left-only / Right-only production-sampled keys, and whether warehouse SQL agrees with Python;
3. a text report (`OUT/<pair>.keys.txt`, plus `.csv`) with every unmatched key, its nearest
   counterpart on the other side, why they differ, and which family would fix it;
4. the match rate of every registered family per key column, so you can try one before editing `CONFIG`.

| Family | Steps | `'007'` | `' 7.50 '` | `'2026-01-31 00:00:00'` |
| --- | --- | --- | --- | --- |
| `string` (default) | trim, lower | `007` | `7.50` | `2026-01-31 00:00:00` |
| `exact` | trim | `007` | `7.50` | `2026-01-31 00:00:00` |
| `numeric` | trim, strip_leading_zeros, collapse_decimal | `7` | `7.5` | unchanged |
| `date` | trim, date | NULL | NULL | `2026-01-31` |
| `digits` | digits | `007` | `750` | `20260131000000` |

A family can also be a step list or an object with different sides, written in `key_families`:
```json
"key_families": {
  "ACCT_ID": "numeric",
  "BRANCH": ["trim", "upper", "strip_prefix:BR-"],
  "EFF_DT": {"family": "date", "left_transform": ["trim", "date"], "right_transform": ["trim", "date"]}
}
```

In [ ]:
CONFIG = "c.json"
PAIR = "orders"
FROM_DATE = None
TO_DATE = None
DAYS = 3
FAMILIES = None
ONLY_KEYS = None
SAMPLES = [
    "007",
    "+7.00",
    " 7.50 ",
    "ABC ",
    "123-45-6789",
    "2026-01-31",
    "20260131",
    "2026-01-31 00:00:00",
    "01/31/2026",
    "",
]
OUT = ".dtrack-local/keycheck"
INLINE = False
MAX_EXAMPLES = 200

`FROM_DATE` / `TO_DATE`: `None` means the last `DAYS` days up to yesterday. `FAMILIES`: `None` uses the
pair's `key_families`; `{"ACCT_ID": "numeric"}` tests a diagnostic override (and therefore a different sample).
`ONLY_KEYS`: `None` checks every sampled key; `[["007", "2026-01-31"]]` narrows the report after
both production sample queries run. `SAMPLES`: values for the self-check in section 1.
`INLINE`: `True` when presigned downloads are blocked.

In [ ]:
import os
from datetime import UTC, date, datetime, timedelta
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
pd.set_option("display.max_columns", 40, "display.width", 200, "display.max_colwidth", 80)

from dtrack_left.backends.databricks import DEFAULT_PROFILE, workspace
from dtrack_left.backends.sources import source_from_config
from dtrack_left.cli.config import load_config
from dtrack_left.common.audit import SqlAudit, attach_sql_audit
from dtrack_local import keycheck
from dtrack_local.pipeline import WarehouseTarget
from dtrack_local.warehouse import Warehouse
from dtrack_right.backends.store import VolumeStore
from dtrack_right.common.hashing import FAMILIES as REGISTERED
from dtrack_right.common.hashing import spark_literal

config = load_config(CONFIG)
pair = config["pairs"][PAIR]
to_date = TO_DATE or (datetime.now(UTC).astimezone().date() - timedelta(days=1)).isoformat()
from_date = FROM_DATE or (date.fromisoformat(to_date) - timedelta(days=DAYS - 1)).isoformat()
work = Path(OUT) / PAIR
work.mkdir(parents=True, exist_ok=True)

client = workspace(os.getenv("DTRACK_DBX_PROFILE") or DEFAULT_PROFILE)
audit_store = VolumeStore(work)
warehouse = Warehouse(client, os.getenv("DTRACK_DBX_WAREHOUSE_ID", ""), inline=INLINE, sql_audit=SqlAudit(audit_store, "right", pair=PAIR))
target = WarehouseTarget(warehouse, pair["right"], work)
source = attach_sql_audit(source_from_config(pair["left"]), SqlAudit(audit_store, "left", pair=PAIR))

left_keys, right_keys, families = keycheck.key_spec(pair, FAMILIES)
print(f"{PAIR}: {from_date}..{to_date}")
pd.DataFrame(
    {
        "left key": left_keys,
        "right key": right_keys,
        "family": [f.spec() for f in families],
        "left_transform": [list(f.left_steps or f.steps) for f in families],
        "right_transform": [list(f.right_steps or f.steps) for f in families],
        "class": [type(f).__name__ for f in families],
    }
)

## 1 · Self-check: Python and warehouse SQL agree on each family

Every value in `SAMPLES` goes through each key's family: `left_transform` in Python,
`right_transform` in Python, and `right_transform_sql` run on the warehouse. `sql_ok` must be `True`
everywhere; a `False` means the Databricks job would hash this value differently from the Left side.
`left == right` shows what an asymmetric family does to the same raw value.

In [ ]:
values = ", ".join(f"({spark_literal(v)})" if v is not None else "(NULL)" for v in SAMPLES)
checks = []
for key, family in zip(left_keys, families):
    sql = f"SELECT v, {family.sql('v', 'right')} FROM VALUES {values} AS t(v)"
    for raw, from_sql in warehouse.rows(sql):
        left_norm, right_norm = family.normalize(raw, "left"), family.normalize(raw, "right")
        checks.append(
            {
                "key": key,
                "raw": repr(raw),
                "left_transform": left_norm,
                "right_transform": right_norm,
                "right_transform_sql": from_sql,
                "left == right": left_norm == right_norm,
                "sql_ok": from_sql == right_norm,
            }
        )
self_check = pd.DataFrame(checks)
print("all SQL results agree with Python:", bool(self_check["sql_ok"].all()))
self_check

## 2 · Run the production sample on both backends

Left and Right both filter in their backend query before CSV output. Fraction mode compiles the same
normalised hash contract into each SQL dialect. Fixed mode writes `fixed.keys.csv` from Left, then uses
those exact normalised tuples in both predicates. Right also returns the Spark-normalised composite so
the report can verify SQL/Python parity.

In [ ]:
left_counts = source.partition_counts(from_date, to_date)
partitions = sorted(left_counts)
print(f"{len(partitions)} Left day(s), {sum(left_counts.values())} row(s):", partitions)
check = keycheck.run_sample_check(
    PAIR,
    pair,
    source,
    target,
    partitions,
    work,
    window=(from_date, to_date),
    families=FAMILIES,
    only_keys=ONLY_KEYS,
    max_examples=MAX_EXAMPLES,
)
SqlAudit(audit_store, "key-check").refresh()
print("sample:", check.sample)
print("fixed keys selected by Left:", len(check.selected_keys))
display(pd.DataFrame.from_dict(check.artifacts, orient="index", columns=["path"]))

## 3 · Compare and write the report

`ok` is `True` when every Left key has a Right counterpart, every Right key has a Left one, and the
warehouse SQL normalised every Right key exactly like Python. Rows missing on one side are still listed;
the report separates them ("no near match: the row itself is missing") from keys written differently.

In [ ]:
report = Path(check.artifacts["report"])
print("report:", report.resolve())
print("csv:   ", report.with_suffix(".csv").resolve())
pd.Series(check.summary())

In [ ]:
print("\n".join(report.read_text().splitlines()[:80]))

## 4 · Which family fits each key column

For every registered family: the share of distinct Left values whose normalised form also exists on the
Right (NULLs never count). `*` marks the family in use. Pick the highest rate, set it in `FAMILIES`, and
re-run section 2 (the family changes sample selection); then write it into `key_families` in `CONFIG`.

In [ ]:
check.suggestions

In [ ]:
display(check.left_only.head(20))
display(check.right_only.head(20))
display(check.sql_mismatch.head(20))

## Adding a family

- **No code (config only):** a step list or an object in `key_families`. The manifest carries the steps,
  so the Databricks job applies the same ones without a redeploy. Steps: `trim`, `lower`, `upper`, `collapse_decimal`, `strip_leading_zeros`, `digits`, `alnum`, `date`, `lpad:N`, `strip_prefix:X`, `strip_suffix:X`.
- **Named family from steps:** `register_family("account", ["trim", "strip_leading_zeros"])` at the end
  of `common/hashing.py`.
- **Custom class:** subclass `KeyFamily` in `common/hashing.py`, override any of `transform`,
  `left_transform`, `right_transform` (Python) and `transform_sql`, `left_transform_sql`,
  `right_transform_sql` (Spark SQL); the side methods default to `transform` / `transform_sql`.
  Decorate it with `@register_family("name")`.

Add Python/CSV behavior to `dtrack_left`; add only the portable family and Spark SQL behavior needed
by `dtrack_right`. The packages are intentionally not byte-identical. Run `just test unit` (behavior
and manifest parity) and `just test spark` (Python vs Spark for every family), then
re-upload `dtrack_right` to the workspace so the job sees the new family. Re-run section 1 to confirm.

In [ ]:
pd.DataFrame(
    [
        {
            "family": name,
            "class": type(f).__name__,
            "transform": list(f.steps),
            "left_transform": list(f.left_steps or f.steps),
            "right_transform": list(f.right_steps or f.steps),
        }
        for name, f in sorted(REGISTERED.items())
    ]
)